In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import statsmodels.api as sm
import plotly.express as px
import nbformat

In [2]:
df = pd.read_csv(r"D:\Masters\Acturial Modelling\umn-2023-travelers-analytics-case-competition\InsNova_data_2023_train.csv")

In [3]:
df

,id,veh_value,exposure,veh_body,veh_age,gender,area,agecat,engine_type,max_power,...,marital_status,e_bill,time_of_week_driven,time_driven,trm_len,credit_score,high_education_ind,clm,numclaims,claimcst0
0,1,0.77,0.444504,SEDAN,4,M,D,3,petrol,147,...,S,1,weekday,6pm - 12am,6,640.448137,1.0,0,0,0.000000
1,2,4.45,0.562183,STNWG,1,M,A,3,petrol,158,...,S,1,weekday,6am - 12pm,12,683.749691,0.0,0,0,0.000000
2,3,4.90,0.465244,STNWG,1,F,A,3,petrol,159,...,M,1,weekday,6pm - 12am,6,653.656117,1.0,0,0,0.000000
3,4,0.48,0.271039,PANVN,4,M,A,4,petrol,80,...,S,1,weekday,12pm - 6pm,12,642.574671,0.0,0,0,0.000000
4,5,0.85,0.141624,SEDAN,4,F,A,5,petrol,126,...,S,0,weekday,6am - 12pm,6,647.175035,0.0,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22614,22615,3.71,0.580943,STNWG,2,F,B,2,petrol,154,...,M,1,weekday,6pm - 12am,12,654.451512,0.0,1,1,280.403348
22615,22616,0.77,0.636641,SEDAN,4,F,C,6,petrol,160,...,M,1,weekday,12pm - 6pm,12,641.163999,0.0,0,0,0.000000
22616,22617,1.95,0.709227,HBACK,1,M,C,6,petrol,146,...,M,1,weekday,12am - 6 am,12,649.644433,0.0,1,2,1253.261110
22617,22618,3.80,0.600221,TRUCK,2,M,A,4,petrol,284,...,S,0,weekday,6pm - 12am,12,653.024119,0.0,0,0,0.000000


In [4]:
# TARGET_COST      = 'claimcst0'   # claim cost
# TARGET_FREQ      = 'numclaims'   # number of claims  
# EXPOSURE         = 'exposure'    # policy duration fraction

In [5]:
# #step1 = response variable/target eda
# The most critical part of EDA. The shape of your target variable determines your model family, link function, and whether to use a one-part or two-part model.

In [6]:
df[df['claimcst0'].isna()] ## no missing values

,id,veh_value,exposure,veh_body,veh_age,gender,area,agecat,engine_type,max_power,...,marital_status,e_bill,time_of_week_driven,time_driven,trm_len,credit_score,high_education_ind,clm,numclaims,claimcst0


In [7]:
df['claimcst0'].describe()

count    22619.000000
mean       163.048084
std       1271.955238
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      57895.584560
Name: claimcst0, dtype: float64

In [8]:
#df['claimcst0'].hist(bins=50)
fig = px.histogram(data_frame=df, x= df['claimcst0'], nbins= 50, title = "Distribution of Raw ClaimCst")
fig.show()

In [9]:
df['claimcst0'].describe()

count    22619.000000
mean       163.048084
std       1271.955238
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      57895.584560
Name: claimcst0, dtype: float64

In [10]:
nonnil_clm = df[df['numclaims']>0]
nonnil_clm

,id,veh_value,exposure,veh_body,veh_age,gender,area,agecat,engine_type,max_power,...,marital_status,e_bill,time_of_week_driven,time_driven,trm_len,credit_score,high_education_ind,clm,numclaims,claimcst0
12,13,1.94,0.610934,HBACK,2,F,C,2,petrol,73,...,M,1,weekday,12am - 6 am,12,641.125045,0.0,1,1,209.047053
48,49,1.79,0.466995,HBACK,2,F,C,3,dissel,70,...,M,1,weekday,12am - 6 am,6,650.545430,1.0,1,1,1586.249277
78,79,0.86,0.256492,HBACK,3,F,C,4,petrol,144,...,S,0,weekday,12pm - 6pm,6,646.207341,0.0,1,1,212.451560
85,86,1.74,0.878268,HDTOP,3,F,A,1,dissel,229,...,S,0,weekday,6am - 12pm,12,639.735985,1.0,1,2,2463.345293
87,88,3.15,0.659426,UTE,1,M,F,4,hybrid,240,...,M,1,weekend,12pm - 6pm,12,659.211301,0.0,1,1,33895.135760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22581,22582,1.90,0.894650,STNWG,4,M,E,3,petrol,140,...,M,1,weekend,6pm - 12am,12,641.927665,1.0,1,1,8566.158611
22583,22584,2.09,0.469576,STNWG,2,M,A,3,petrol,175,...,M,1,weekday,12pm - 6pm,6,648.385396,1.0,1,1,239.981071
22587,22588,1.43,0.058077,HBACK,2,M,B,6,petrol,87,...,M,1,weekday,6am - 12pm,6,666.045217,0.0,1,1,685.432264
22614,22615,3.71,0.580943,STNWG,2,F,B,2,petrol,154,...,M,1,weekday,6pm - 12am,12,654.451512,0.0,1,1,280.403348


In [11]:
nonnil_clm['claimcst0'].describe()

count     1542.000000
mean      2391.689109
std       4290.988757
min        200.037808
25%        397.497764
50%        877.454329
75%       2408.518547
max      57895.584560
Name: claimcst0, dtype: float64

In [12]:
#distribution of non-nil clmcst
fig2 = px.histogram(data_frame=nonnil_clm, x=nonnil_clm['claimcst0'], nbins=100, title="Distribution of NonNil ClaimCst")
fig2.show()

In [13]:
#as we can see that majority od our data is concentrated at 0 or just above 0 and then the tail is too much elongated towards the right, hence we can take log of this variable
# that will squeeze the right tail in and we can have a better look at our data. 
# Big values come closer to medium values; medium values stay where they are; small values barely move.

In [14]:
nonnil_clm['logClaimCst'] = np.log(nonnil_clm['claimcst0'])

In [15]:
fig= px.histogram(data_frame=nonnil_clm, x = nonnil_clm['logClaimCst'], nbins=75, title="Distribution of LogNonNilClaimCst")
fig.show()

In [16]:
# for severity modelling we have primary 5 distributions
#normal - support(range it covers): all real numbers -ve to +ve ; shape: bell shaped --- not true in this case
#lognormal - support(range it covers): strictly +ve ; shape: right skewed, after log exactly bell shaped --- not true in this case
#gamma - support(range it covers): strictly +ve ; shape: right skewed, after log roughly bell shaped --- holds true in this case
#paretto - support(range it covers): above some threshold ; shape: pure power law tail, very heavy


In [17]:
##looking at the histogram of target var i can confirm the following
# 1) right skewed -- long right tail
# 2) majority of the values are 0
# 3) always positive
# 4) continous

##will use tweedie distribution and not gamma because we have majority of claimCst = 0, gamma needs data > 0, it doesn't handle data=0
#if we are building the freq x severity model then, we can filter our data for claimCst>0 and the fit gamma distribution to model severity 
# but if we want to use a single model only, then we can just directly model using tweedie


#approach A
# building separate freq x severity models, starting with severity.
#approach B
# building one model and modelling claimCst using tweedie distribution

In [18]:
##Approach A -- building severity model